# EV Purchase Prediction — CatBoost + LightGBM + XGBoost Blend

10-fold OOF training, rank-averaged blend, GPU-ready.

In [ ]:
import os
import glob
import sys
import warnings
import importlib.util
import subprocess

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

print("Python version:", sys.version)

In [ ]:
for pkg in ["catboost", "lightgbm", "xgboost"]:
    if importlib.util.find_spec(pkg) is None:
        print(f"{pkg} not found. Installing...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
    else:
        print(f"{pkg} is already installed.")

GPU_AVAILABLE = False
try:
    out = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
    if out.returncode == 0:
        GPU_AVAILABLE = True
        print("GPU detected.")
    else:
        print("No GPU detected. Running on CPU.")
except FileNotFoundError:
    print("No nvidia-smi on PATH. Running on CPU.")

print("GPU_AVAILABLE =", GPU_AVAILABLE)

In [ ]:
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier, early_stopping
from xgboost import XGBClassifier

print("All models imported.")

In [ ]:
def find_file(name_keywords):
    for root, dirs, files in os.walk('/kaggle/input'):
        for f in files:
            low = f.lower()
            if all(k in low for k in name_keywords) and low.endswith('.csv'):
                return os.path.join(root, f)
    return None

train_path = find_file(["train"])
test_path = find_file(["test"])
sample_path = find_file(["sample"])

if train_path is None:
    train_path = "train.csv"
if test_path is None:
    test_path = "test.csv"
if sample_path is None:
    sample_path = "sample_submission.csv"

print("Train:", train_path)
print("Test:", test_path)
print("Sample:", sample_path)

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sample_sub = pd.read_csv(sample_path)

print("\nTrain:", train.shape)
print("Test:", test.shape)
print("Sample:", sample_sub.shape)

In [ ]:
target_col = "Will_Buy_EV"
train[target_col] = train[target_col].astype(str).str.strip().str.lower()
y = train[target_col].map({"yes": 1, "no": 0}).astype(int)
print(y.value_counts())

In [ ]:
id_col = "id"
test_id_col = id_col if id_col in test.columns else test.columns[0]

base_cat = ["Gender", "City_Type", "Current_Car_Type",
            "Range_Anxiety_Level", "Home_Charging_Possible", "Subsidy_Available"]
cat_features = [c for c in base_cat if c in train.columns and c in test.columns]

for c in cat_features:
    train[c] = train[c].astype(str).str.strip().str.lower().replace({"nan":"unknown","":"unknown"})
    test[c]  = test[c].astype(str).str.strip().str.lower().replace({"nan":"unknown","":"unknown"})

print("Base categoricals:", cat_features)

In [ ]:
def add_features(df):
    df = df.copy()
    df["charging_total"] = df["Charging_Stations_Near_Home"] + df["Charging_Stations_Near_Work"]
    df["charging_diff"] = df["Charging_Stations_Near_Home"] - df["Charging_Stations_Near_Work"]
    df["charging_min"] = df[["Charging_Stations_Near_Home","Charging_Stations_Near_Work"]].min(axis=1)
    df["charging_max"] = df[["Charging_Stations_Near_Home","Charging_Stations_Near_Work"]].max(axis=1)
    df["charging_zero"] = ((df["Charging_Stations_Near_Home"]==0) & (df["Charging_Stations_Near_Work"]==0)).astype(int)
    df["income_per_car"] = df["Annual_Income_USD"] / (df["Number_of_Cars_Owned"] + 1)
    df["commute_per_car"] = df["Daily_Commute_km"] / (df["Number_of_Cars_Owned"] + 1)
    df["charging_per_commute"] = df["charging_total"] / (df["Daily_Commute_km"] + 1)
    df["income_per_age"] = df["Annual_Income_USD"] / (df["Age"] + 1)
    rmap = {"low":0, "medium":1, "high":2}
    df["range_anxiety_ord"] = df["Range_Anxiety_Level"].map(rmap).fillna(0).astype(int)
    df["income_x_env"] = df["Annual_Income_USD"] * df["Environmental_Concern_Level"]
    df["commute_x_range"] = df["Daily_Commute_km"] * df["range_anxiety_ord"]
    df["income_is_min"] = (df["Annual_Income_USD"] == 30000).astype(int)
    df["commute_is_5"] = (df["Daily_Commute_km"] == 5.0).astype(int)
    df["home_subsidy"] = df["Home_Charging_Possible"] + "_" + df["Subsidy_Available"]
    df["city_car"] = df["City_Type"] + "_" + df["Current_Car_Type"]
    df["gender_city"] = df["Gender"] + "_" + df["City_Type"]
    df["home_city"] = df["Home_Charging_Possible"] + "_" + df["City_Type"]
    df["subsidy_range"] = df["Subsidy_Available"] + "_" + df["Range_Anxiety_Level"]
    df["env_range"] = df["Environmental_Concern_Level"].astype(str) + "_" + df["Range_Anxiety_Level"]
    return df

train = add_features(train)
test = add_features(test)

new_cats = ["home_subsidy","city_car","gender_city","home_city","subsidy_range","env_range"]
for c in new_cats:
    train[c] = train[c].astype(str)
    test[c] = test[c].astype(str)
cat_features += new_cats

print("Total categoricals:", len(cat_features))

In [ ]:
feature_cols = [c for c in train.columns if c not in [id_col, target_col]]
X = train[feature_cols].copy()
X_test = test[feature_cols].copy()

for c in feature_cols:
    if c in cat_features:
        X[c] = X[c].astype(str).fillna("unknown")
        X_test[c] = X_test[c].astype(str).fillna("unknown")
    else:
        if X[c].isna().any() or X_test[c].isna().any():
            m = X[c].median()
            X[c] = X[c].fillna(m)
            X_test[c] = X_test[c].fillna(m)

print("X:", X.shape, "| X_test:", X_test.shape)

In [ ]:
X_lgb = X.copy()
X_test_lgb = X_test.copy()
for c in cat_features:
    cats = sorted(set(X[c].astype(str)) | set(X_test[c].astype(str)))
    X_lgb[c] = pd.Categorical(X[c].astype(str), categories=cats)
    X_test_lgb[c] = pd.Categorical(X_test[c].astype(str), categories=cats)

X_xgb = X_lgb.copy()
X_test_xgb = X_test_lgb.copy()
print("LGB/XGB copies ready.")

In [ ]:
N_SPLITS = 10
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

oof_cat = np.zeros(len(X))
test_cat = np.zeros(len(X_test))

cat_params = dict(
    iterations=5000, learning_rate=0.03, depth=8, l2_leaf_reg=3.0,
    loss_function="Logloss", eval_metric="AUC",
    od_type="Iter", od_wait=200, auto_class_weights=None,
    max_ctr_complexity=6, one_hot_max_size=10, random_strength=1.0,
    random_seed=42, verbose=200,
)
if GPU_AVAILABLE:
    cat_params.update(task_type="GPU", devices="0",
                      bootstrap_type="Bernoulli", subsample=0.8, border_count=128)
else:
    cat_params["thread_count"] = -1

for fold, (tr, va) in enumerate(skf.split(X, y), 1):
    print(f"\n===== CatBoost Fold {fold}/{N_SPLITS} =====")
    m = CatBoostClassifier(**cat_params)
    m.fit(X.iloc[tr], y.iloc[tr], cat_features=cat_features,
          eval_set=(X.iloc[va], y.iloc[va]), use_best_model=True)
    oof_cat[va] = m.predict_proba(X.iloc[va])[:, 1]
    test_cat += m.predict_proba(X_test)[:, 1] / N_SPLITS
    print(f"Fold {fold} AUC: {roc_auc_score(y.iloc[va], oof_cat[va]):.5f}")
    np.save("oof_cat.npy", oof_cat)
    np.save("test_cat.npy", test_cat)

print(f"\nCatBoost OOF AUC: {roc_auc_score(y, oof_cat):.5f}")

In [ ]:
lgb_params = dict(
    objective="binary", metric="auc", n_estimators=2500,
    learning_rate=0.05, num_leaves=63, max_depth=-1, min_child_samples=30,
    feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1,
    lambda_l1=0.0, lambda_l2=1.0, random_state=42, n_jobs=-1, verbose=-1,
)

oof_lgb = np.zeros(len(X))
test_lgb = np.zeros(len(X_test))

for fold, (tr, va) in enumerate(skf.split(X_lgb, y), 1):
    print(f"\n===== LightGBM Fold {fold}/{N_SPLITS} =====")
    m = LGBMClassifier(**lgb_params)
    m.fit(X_lgb.iloc[tr], y.iloc[tr], eval_set=[(X_lgb.iloc[va], y.iloc[va])],
          eval_metric="auc", categorical_feature=cat_features,
          callbacks=[early_stopping(150)])
    oof_lgb[va] = m.predict_proba(X_lgb.iloc[va])[:, 1]
    test_lgb += m.predict_proba(X_test_lgb)[:, 1] / N_SPLITS
    print(f"Fold {fold} AUC: {roc_auc_score(y.iloc[va], oof_lgb[va]):.5f}")
    np.save("oof_lgb.npy", oof_lgb)
    np.save("test_lgb.npy", test_lgb)

print(f"\nLightGBM OOF AUC: {roc_auc_score(y, oof_lgb):.5f}")

In [ ]:
xgb_params = dict(
    n_estimators=2500, learning_rate=0.05, max_depth=6, min_child_weight=5,
    subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
    objective="binary:logistic", eval_metric="auc",
    tree_method="hist", device="cuda" if GPU_AVAILABLE else "cpu",
    enable_categorical=True, random_state=42, n_jobs=-1,
)

oof_xgb = np.zeros(len(X))
test_xgb = np.zeros(len(X_test))

for fold, (tr, va) in enumerate(skf.split(X_xgb, y), 1):
    print(f"\n===== XGBoost Fold {fold}/{N_SPLITS} =====")
    m = XGBClassifier(**xgb_params)
    m.fit(X_xgb.iloc[tr], y.iloc[tr],
          eval_set=[(X_xgb.iloc[va], y.iloc[va])], verbose=False)
    oof_xgb[va] = m.predict_proba(X_xgb.iloc[va])[:, 1]
    test_xgb += m.predict_proba(X_test_xgb)[:, 1] / N_SPLITS
    print(f"Fold {fold} AUC: {roc_auc_score(y.iloc[va], oof_xgb[va]):.5f}")
    np.save("oof_xgb.npy", oof_xgb)
    np.save("test_xgb.npy", test_xgb)

print(f"\nXGBoost OOF AUC: {roc_auc_score(y, oof_xgb):.5f}")

In [ ]:
from scipy.optimize import minimize
from scipy.stats import rankdata

r_cat = rankdata(oof_cat) / len(oof_cat)
r_lgb = rankdata(oof_lgb) / len(oof_lgb)
r_xgb = rankdata(oof_xgb) / len(oof_xgb)

def neg_auc(w):
    w = np.clip(w, 0, 1)
    s = w.sum()
    if s == 0: return 0.0
    w = w / s
    return -roc_auc_score(y, w[0]*r_cat + w[1]*r_lgb + w[2]*r_xgb)

res = minimize(neg_auc, x0=[0.4, 0.3, 0.3], method="Nelder-Mead",
               options={"xatol": 1e-4, "fatol": 1e-6})
best_w = np.clip(res.x, 0, 1); best_w = best_w / best_w.sum()
print("Weights:", best_w)

blend_oof = best_w[0]*r_cat + best_w[1]*r_lgb + best_w[2]*r_xgb
print("Blend OOF AUC:", roc_auc_score(y, blend_oof))

tr_cat = rankdata(test_cat) / len(test_cat)
tr_lgb = rankdata(test_lgb) / len(test_lgb)
tr_xgb = rankdata(test_xgb) / len(test_xgb)
final_pred = best_w[0]*tr_cat + best_w[1]*tr_lgb + best_w[2]*tr_xgb

In [ ]:
sub_id_col = sample_sub.columns[0]
sub_target_col = sample_sub.columns[1]

pred_df = pd.DataFrame({test_id_col: test[test_id_col].values, sub_target_col: final_pred})
submission = sample_sub[[sub_id_col]].merge(pred_df,
                                            left_on=sub_id_col,
                                            right_on=test_id_col, how="left")
if test_id_col != sub_id_col and test_id_col in submission.columns:
    submission = submission.drop(columns=[test_id_col])
submission = submission[[sub_id_col, sub_target_col]]

print("Shape:", submission.shape)
print("Missing:", int(submission[sub_target_col].isna().sum()))

In [ ]:
submission.to_csv("submission.csv", index=False)
print("Saved submission.csv")
print(submission.head())
print(submission.shape)